# load packages

In [ ]:
from scipy.sparse import coo_matrix
import numpy as np
import scipy as sp
import pandas as pd

To set up access token, see [here](https://connectome-neuprint.github.io/neuprint-python/docs/quickstart.html).

In [ ]:
from neuprint import (
    Client,
    fetch_adjacencies,
    fetch_neurons,
    NeuronCriteria as NC,
)

c = Client("neuprint.janelia.org", dataset="hemibrain:v1.2.1")

In [ ]:
neuron_df, conn_df = fetch_adjacencies(min_total_weight=1)
neuron_df

In [ ]:
neuron_df = neuron_df[neuron_df["type"].notna()]
neuron_df.shape[0] ** 2 * 8 / 1e9  # GB

In [ ]:
# take last two letters of instance
neuron_df.loc[:, ["side"]] = neuron_df.instance.str[-2:]
neuron_df.loc[:, ["side"]] = neuron_df.side.str.replace("_L", "left")
neuron_df.loc[:, ["side"]] = neuron_df.side.str.replace("_R", "right")
# if it's not left or right, set to 'unknown'
neuron_df.loc[~neuron_df.side.isin(["left", "right"]), ["side"]] = "noside"
neuron_df

In [ ]:
conn_df = conn_df[
    conn_df.bodyId_pre.isin(neuron_df.bodyId)
    & conn_df.bodyId_post.isin(neuron_df.bodyId)
]

In [ ]:
conn = conn_df.groupby(["bodyId_pre", "bodyId_post"]).weight.sum().reset_index()
conn

In [ ]:
# instead of making a dense matrix based on the edgelist above, let's make a sparse one from the edgelist directly
# first make a coo matrix
nodes = set(neuron_df.bodyId)
sorted_nodes = sorted(nodes)  # Convert the set to a sorted list
nodes_to_idx = {node: num for num, node in enumerate(sorted_nodes)}

# type to type connectivity
conn["pre_idx"] = conn.bodyId_pre.map(nodes_to_idx)
conn["post_idx"] = conn.bodyId_post.map(nodes_to_idx)

# Create COO matrix
row = conn["pre_idx"].values
col = conn["post_idx"].values
data = conn["weight"].values
matrix_size = len(nodes)
coo = coo_matrix((data, (row, col)), shape=(matrix_size, matrix_size))

# then turn it into csc matrix
csc = coo.tocsc()

# calculate the size
csc_size = csc.data.nbytes  # Size of the data array
csc_size += csc.indices.nbytes  # Size of the indices array
csc_size += csc.indptr.nbytes  # Size of the index pointer array
# number of MB
csc_size / 1e6

In [ ]:
csc = csc.astype(np.int16)

# calculate the size
csc_size = csc.data.nbytes  # Size of the data array
csc_size += csc.indices.nbytes  # Size of the indices array
csc_size += csc.indptr.nbytes  # Size of the index pointer array
# number of MB
csc_size / 1e6

In [ ]:
sp.sparse.save_npz("../data/hemibrain_neuron.npz", csc)

In [ ]:
meta = neuron_df[["bodyId", "type", "side"]].copy()
meta.rename(columns={"type": "cell_type"}, inplace=True)
meta["idx"] = meta["bodyId"].map(nodes_to_idx)

meta.to_csv("../data/hemibrain_neuron.csv", index=False)
meta